# Figure 1: ZnPc optical tensor at the carbon K-edge (ML-DFT vs. experiment)

Compares the ML-DFT and experimentally derived uniaxial optical tensors of ZnPc over 281–290 eV: absorptive (β) and dispersive (δ) components with the isotropic average, probe energies marked, and a normalized DFT-vs-experiment deviation.

**Inputs** (not distributed; available from the authors on request, see `README.md`): `read_ooc("dft.csv")`, `read_ooc("exp.csv")` → `@models/optical/znpc/`.

**Outputs:** `FIG1.png`, `FIG1.pdf`, `fig1_dials_probe_energies.{png,pdf}`.

**Run:** cells in order; seconds on a laptop.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.interpolate import Akima1DInterpolator

from utils import read_ooc
from utils.helpers.plotting_helper import set_plotting_defaults

set_plotting_defaults()

In [ ]:
dft = read_ooc("dft.csv", material="znpc")
exp = read_ooc("exp.csv", material="znpc")
# crop to 275 eV - 300 eV
# pxr energies
pxr = [250.0,
 275.0,
 280.0,
 281.0,
 281.2,
 281.4,
 281.6,
 281.8,
 282.0,
 282.3,
 282.5,
 282.7,
 282.9,
 283.1,
 283.3,
 283.5,
 283.7,
 284.2,
 285.1,
 287.0,
 289.0]

In [ ]:
"""
Figure 1: ZnPc index-of-refraction tensor across the carbon K-edge.

Panels
  (a)  beta components (n_ixx, n_izz), exp markers vs DFT curves
  (b)  delta components (n_xx, n_zz), with component-resolved zero
       crossings and the isotropic average

Annotation channels (one encoding each)
  dashed purple vline   283.7 eV long-q probe, spanning both panels;
                        the cross fill terminates against it
  tick comb             sparse PXR probe energies (> 283.7 eV) on the
                        top spine of (a)
  grey bars             local n_xx -> n_zz span, only at the sparse
                        probe energies above 283.7 eV
  gold fill             delta_xx -> delta_zz splitting through the
                        fine-grid pre-edge, panel (b)
  "|" ticks             first delta zero crossing per component, on the
                        delta = 0 line, color-matched + labeled
  arrow                 off-axis 250 eV non-resonant deep probe

Prerequisites: dft, exp (read_ooc frames), pxr energy list,
Akima1DInterpolator imported.
"""

import matplotlib.patheffects as pe
import matplotlib.transforms as mtransforms

STROKE = [pe.Stroke(linewidth=1.5, foreground="white"), pe.Normal()]

ub = 290
lb = 281

FINE_LO, FINE_HI = 281.0, 283.5   # 0.2 eV sampling window (reference only)
E_HIQ = 283.7                     # long-q probe energy; cross-fill upper edge

# ── Visual-variant flags ─────────────────────────────────────────────────────
SHOW_SPAN_BARS = True    # grey n_xx -> n_zz bars at sparse probe energies
ANNOTATE_EPS = True      # compact eps stats block in panel (b)

PURPLE = "purple"

# Pre-edge cross fill, panel (b). Soft gold separates cleanly from C0/C1.
FILL_COLOR = "#D9A441"
FILL_ALPHA = 0.30
FILL_LBL = (283.0, 0.45)          # (data x, axes-fraction y); nudge
FILL_LBL_COLOR = "#A87B1F"

# Direct curve-label positions (axes fraction); nudge per final render.
CURVE_LBL = {
    "iso":  (0.95, 0.2, "0.45"),
}

# Crossing-label offsets in display points from each "|" tick; nudge.
CROSS_LBL_OFF = {
    "n_xx": (0, -15),
    "n_zz": (0, -15),
}

# Long-q label position, panel (a): (axes-fraction y, points x-offset); nudge.
HIQ_LBL_Y = 0.97
HIQ_LBL_DX = 2.5

# Eps stats block anchor (axes fraction, panel b); nudge per final render.
EPS_LBL = (0.97, 0.04)

dft = dft[(dft["energy"] >= lb) & (dft["energy"] <= ub)]
exp = exp[(exp["energy"] >= lb) & (exp["energy"] <= ub)]


def density_scale(component, density):
    return component * density

def rotate_tensor(xx, zz, gamma):
    xx_r = (xx * (1+np.cos(gamma)**2) + zz * np.sin(gamma)**2) / 2
    zz_r = xx * np.sin(gamma)**2 + zz * np.cos(gamma)**2
    return xx_r, zz_r

def tensor_deviation(dft_df, exp_df, lo, hi,
                     components=("n_xx", "n_zz", "n_ixx", "n_izz")):
    """
    Normalized DFT-vs-exp tensor deviation on a common energy grid.

    Each component difference is scaled by the maximum magnitude of the
    experimental component over the window (delta crosses zero, so a
    pointwise relative error is ill-defined there). epsilon(E) is the RMS
    over the four scaled components, in percent. Returns the grid, the
    combined epsilon, and the per-component signed deviations.
    """
    grid = np.linspace(lo, hi, 1500)
    eps_sq = np.zeros_like(grid)
    per = {}
    for c in components:
        d = np.interp(grid, dft_df["energy"], dft_df[c])
        x = np.interp(grid, exp_df["energy"], exp_df[c])
        scale = float(np.max(np.abs(x)))
        per[c] = 100.0 * (d - x) / scale
        eps_sq += per[c] ** 2
    eps = np.sqrt(eps_sq / len(components))
    return grid, eps, per


def zero_crossings(e, y):
    """Linear-interpolated zero crossings of y(e); returns a list of energies."""
    e = np.asarray(e, dtype=float)
    y = np.asarray(y, dtype=float)
    idx = np.where(np.diff(np.sign(y)) != 0)[0]
    return [
        e[j] - y[j] * (e[j + 1] - e[j]) / (y[j + 1] - y[j]) for j in idx
    ]


fig_width = 3.35
fig_height = 4.5
fig, (ax1, ax2) = plt.subplots(
    2, 1,
    figsize=(fig_width, fig_height),
    height_ratios=[1, 1],
    gridspec_kw={"hspace": 0.1},
    sharex=True,
    dpi=300,
)

# ── Panel (a): beta components ───────────────────────────────────────────────

ax1.plot(
    exp["energy"],
    exp["n_ixx"],
    "o",
    label=r"$n_{xx}$, exp",
    markeredgecolor="black",
    markerfacecolor="white",
    linestyle="none",
    markersize=4,
    markeredgewidth=1,
    markevery=10,
)
ax1.plot(
    exp["energy"],
    exp["n_izz"],
    "^",
    label=r"$n_{zz}$, exp",
    markeredgecolor="black",
    markerfacecolor="white",
    linestyle="none",
    markersize=4,
    markeredgewidth=1,
    markevery=10,
)
ax1.plot(
    dft["energy"],
    dft["n_ixx"],
    "-",
    color="C0",
    linewidth=1,
    label=r"$n_{xx}$, ML-DFT",
)
ax1.plot(
    dft["energy"],
    dft["n_izz"],
    "-",
    color="C1",
    linewidth=1,
    label=r"$n_{zz}$, ML-DFT",
)
ax1_ylim = ax1.get_ylim()
# Define a tighter gradation: mostly close to 1, some further out
# density_gradation = np.array([1.0, 1.01, 1.03, 1.08, 1.15, 1.2, 1.3])
# center = 1.0
# for density in density_gradation:
#     # Alpha gets lighter the further from 1.0
#     alpha = max(0.05, 0.55 - abs(density - center)*2)
#     ixx_d = density_scale(dft["n_ixx"], density)
#     izz_d = density_scale(dft["n_izz"], density)
#     ax1.plot(
#         dft["energy"],
#         ixx_d,
#         "-",
#         color="C0",
#         linewidth=0.6,
#         # zorder=-10,
#         alpha=alpha,
#     )
#     ax1.plot(
#         dft["energy"],
#         izz_d,
#         "-",
#         color="C1",
#         linewidth=0.6,
#         # zorder=-10,
#         alpha=alpha,
#     )
# Reset the y-limits to the original values
ax1.set_ylim(ax1_ylim)

ax1.plot(
    dft["energy"],
    2 * dft["n_ixx"] / 3 + dft["n_izz"] / 3,
    "-",
    color="grey",
    linewidth=0.5,
    zorder=-10,
)

ax1.set_ylabel(r"$\beta$ (a.u.)")
ax1.set_xlim(lb, ub)
ax1.yaxis.offsetText.set_fontsize(10)
ax1.tick_params(axis="both", which="major", labelsize=11)
ax1.ticklabel_format(style="sci", axis="y", scilimits=(0, 0))
# Matrix legend: rows = tensor component, columns = source. Column widths
# follow the rendered header text so labels never collide; units are points.
from matplotlib.font_manager import FontProperties
from matplotlib.lines import Line2D
from matplotlib.offsetbox import AnchoredOffsetbox, DrawingArea
from matplotlib.text import Text
from matplotlib.textpath import TextPath

LEG_FS = 7                       # header font size (pt); row labels +0.5
LEG_ANCHOR = (0.985, 0.975)      # upper-right corner, axes fraction; nudge
LEG_ROW_H, LEG_HEAD_H = 9.5, 10  # row and header heights (pt)
LEG_GLYPH_W, LEG_GAP = 12, 4     # line-glyph length, column padding (pt)

leg_cols = ["NEXAFS", "ML-DFT"]
leg_rows = [r"$n_{xx}$", r"$n_{zz}$"]
leg_cells = [[("o", None), (None, "C0")],
             [("^", None), (None, "C1")]]   # (marker, line color)


def _text_w(s, fs):
    return TextPath((0, 0), s, size=fs, prop=FontProperties()).get_extents().width


row_w = max(_text_w(t, LEG_FS + 0.5) for t in leg_rows) + LEG_GAP
col_w = [max(_text_w(t, LEG_FS), LEG_GLYPH_W) + LEG_GAP for t in leg_cols]
leg_W = row_w + sum(col_w)
leg_H = LEG_HEAD_H + len(leg_rows) * LEG_ROW_H
leg_da = DrawingArea(leg_W, leg_H, 0, 0, clip=False)

col_x = row_w + np.cumsum([0] + col_w[:-1]) + np.array(col_w) / 2
for x, t in zip(col_x, leg_cols):
    leg_da.add_artist(Text(x, leg_H - LEG_HEAD_H / 2, t, ha="center",
                           va="center", fontsize=LEG_FS))
for i, t in enumerate(leg_rows):
    y = leg_H - LEG_HEAD_H - (i + 0.5) * LEG_ROW_H
    leg_da.add_artist(Text(row_w - LEG_GAP / 2, y, t, ha="right", va="center",
                           fontsize=LEG_FS + 0.5))
    for x, (mk, lc) in zip(col_x, leg_cells[i]):
        if mk is not None:
            leg_da.add_artist(Line2D(
                [x], [y], marker=mk, ls="none", mfc="white", mec="black",
                mew=1.0, ms=4,
            ))
        else:
            leg_da.add_artist(Line2D(
                [x - LEG_GLYPH_W / 2, x + LEG_GLYPH_W / 2], [y, y],
                color=lc, lw=1.2, solid_capstyle="butt",
            ))

leg_box = AnchoredOffsetbox(
    loc="upper right", child=leg_da, frameon=False, pad=0.0, borderpad=0.0,
    bbox_to_anchor=LEG_ANCHOR, bbox_transform=ax1.transAxes,
)
leg_box.set_zorder(10)
ax1.add_artist(leg_box)

# Direct curve labels (component color carries to panel b).
for key, (xf, yf, col) in CURVE_LBL.items():
    txt = {"n_zz": r"$n_{zz}$", "n_xx": r"$n_{xx}$", "iso": r"$\bar{n}$"}[key]
    ax1.text(
        xf, yf, txt, transform=ax1.transAxes,
        color=col, fontsize=8,
        ha="right" if key == "n_zz" else "left",
        path_effects=STROKE,
    )

# Off-axis 250 eV non-resonant deep probe.
ax1.annotate(
    r"250 eV", xy=(0.012, 0.1), xytext=(0.115, 0.1),
    xycoords="axes fraction", textcoords="axes fraction",
    ha="left", va="center", fontsize=7, color="0.25",
    arrowprops=dict(arrowstyle="-|>", color="0.25", lw=0.8),
)

# ── Panel (b): delta components ──────────────────────────────────────────────

ax2.plot(
    exp["energy"],
    exp["n_xx"],
    "o",
    label=None,
    markeredgecolor="black",
    markerfacecolor="white",
    linestyle="none",
    markersize=4,
    markeredgewidth=1,
    markevery=10,
)
ax2.plot(
    exp["energy"],
    exp["n_zz"],
    "^",
    label=None,
    markeredgecolor="black",
    markerfacecolor="white",
    linestyle="none",
    markersize=4,
    markeredgewidth=1,
    markevery=10,
)
ax2.plot(dft["energy"], dft["n_xx"], "-", color="C0", linewidth=1, label=None)
ax2.plot(dft["energy"], dft["n_zz"], "-", color="C1", linewidth=1, label=None)

iso_delta = 2 * dft["n_xx"] / 3 + dft["n_zz"] / 3
ax2.plot(
    dft["energy"], iso_delta, "-",
    color="grey", linewidth=0.5, zorder=-10,
)
ax2.axhline(
    0,
    color=plt.rcParams["axes.edgecolor"],
    lw=plt.rcParams["axes.linewidth"],
)
ax2.ticklabel_format(style="scientific", axis="y", scilimits=(0, 0))
ax2.set_ylabel(r"$\delta$ (a.u.)")
ax2.set_xlabel("Energy (eV)")

# orientation_gradation = np.deg2rad([0, .5, 2, 5, 10, 20, 30, 40,50])
# for orientation in orientation_gradation:
#     xx_r, zz_r = rotate_tensor(dft["n_xx"], dft["n_zz"], orientation)
#     alpha = max(0.05, 0.55 - abs(orientation - center)*2)
#     ax2.plot(
#         dft["energy"],
#         xx_r,
#         "-",
#         color="C0",
#         linewidth=.5,
#         # zorder=-10,
#         alpha=alpha,
#     )
#     ax2.plot(
#         dft["energy"],
#         zz_r,
#         "-",
#         color="C1",
#         linewidth=.5,
#         # zorder=-10,
#         alpha=alpha,
#     )

# ── Long-q probe: one encoding, both panels ──────────────────────────────────
# Dashed purple vline at 283.7 eV; the cross fill terminates against it.
# Replaces the purple comb tick and the purple span bars.

for ax in (ax1, ax2):
    ax.axvline(
        E_HIQ, color=PURPLE, lw=0.8, ls=(0, (3, 2)), alpha=0.9, zorder=-5,
    )
ax1.annotate(
    f"{E_HIQ} eV",
    xy=(E_HIQ, HIQ_LBL_Y),
    xycoords=mtransforms.blended_transform_factory(ax1.transData, ax1.transAxes),
    xytext=(HIQ_LBL_DX, 0), textcoords="offset points",
    rotation=90, ha="left", va="top",
    fontsize=6.5, color=PURPLE, path_effects=STROKE, zorder=6,
)

# ── Delta zero crossings, component resolved ─────────────────────────────────
# First crossing only, one per component. Vertical "|" ticks through the
# delta = 0 line; no other element in the figure uses this glyph, so they
# cannot be confused with exp data markers. Labels nudge via CROSS_LBL_OFF
# (display points). All crossings are still printed for the text.

cross = {}
for key, ydata, color, lbl in (
    ("n_xx", dft["n_xx"], "C0", r"$E^{0}_{xx}$"),
    ("n_zz", dft["n_zz"], "C1", r"$E^{0}_{zz}$"),
):
    cs = zero_crossings(dft["energy"], ydata)
    cross[key] = cs
    if cs:
        ax2.plot(
            cs[0], 0, marker="|", ms=8, mec=color, mew=1.4,
            linestyle="none", zorder=6,
        )
        ax2.annotate(
            lbl, xy=(cs[0], 0),
            xytext=CROSS_LBL_OFF[key], textcoords="offset points",
            color=color, fontsize=7, ha="left", va="bottom",
            path_effects=STROKE, zorder=6,
        )
        print(
            f"delta {key} zero crossing(s): "
            + ", ".join(f"{c:.2f}" for c in cs) + " eV"
        )
print(
    "isotropic delta crossing(s): "
    + ", ".join(f"{c:.2f}" for c in zero_crossings(dft["energy"], iso_delta))
    + " eV"
)

# Pre-edge cross fill: delta_xx -> delta_zz splitting through the fine-grid
# window, terminating at the long-q line. Carries both the fine-sampling
# message and the tensor-splitting magnitude; labeled directly.
_e = dft["energy"].to_numpy()
_m = _e <= E_HIQ
ax2.fill_between(
    _e[_m], dft["n_xx"].to_numpy()[_m], dft["n_zz"].to_numpy()[_m],
    color=FILL_COLOR, alpha=FILL_ALPHA, lw=0, zorder=-15,
)
ax2.text(
    FILL_LBL[0], FILL_LBL[1], r"$\Delta\delta$",
    transform=mtransforms.blended_transform_factory(ax2.transData, ax2.transAxes),
    color=FILL_LBL_COLOR, fontsize=7, ha="center", va="center",
    path_effects=STROKE, zorder=4,
)

# ── Probe-energy indicators ──────────────────────────────────────────────────

def at(df, val, col):
    """
    Interpolates the values along column `col` of dataframe `df` using
    Akima1DInterpolator (makima),
    and returns the interpolated value at position `val`.
    """
    x = df[col]
    y = df.drop(columns=[col])
    interp = Akima1DInterpolator(x, y.values, axis=0)
    return interp(val)


# Tick comb hanging from the top spine of (a): sparse probe energies only.
# The pre-edge fine grid is carried by the fill, the long-q energy by the
# dashed vline, so neither appears in the comb.
comb_tr = mtransforms.blended_transform_factory(ax1.transData, ax1.transAxes)
for en in pxr:
    if en <= E_HIQ or en > ub:
        continue
    ax1.plot(
        [en, en], [1.0, 0.955],
        transform=comb_tr,
        color="0.3",
        lw=0.8,
        solid_capstyle="butt",
        clip_on=False,
        zorder=5,
    )

# Span bars: local n_xx -> n_zz tensor splitting, only at the sparse probe
# energies above the long-q line. Below it the fill carries the span, so
# bars there would just fence the shading.
if SHOW_SPAN_BARS:
    for en in pxr:
        if en <= E_HIQ or en > ub:
            continue
        val = at(dft, en, "energy")
        ax1.vlines(en, val[-2], val[-1], color="0.65", lw=0.8, zorder=-10)
        ax2.vlines(en, val[0], val[1], color="0.65", lw=0.8, zorder=-10)

# ── Eps stats block, panel (b) ───────────────────────────────────────────────
# Compact presentation of the total and peak percent error; no pointer.

grid, eps, per = tensor_deviation(dft, exp, lb, ub)
_imax = int(np.argmax(eps))
if ANNOTATE_EPS:
    ax2.text(
        *EPS_LBL,
        rf"$\langle\varepsilon\rangle = {eps.mean():.1f}\%$"
        "\n"
        rf"$\varepsilon_{{\max}} = {eps[_imax]:.1f}\%$",
        transform=ax2.transAxes,
        fontsize=7, color="0.25", ha="right", va="bottom",
        linespacing=1.3, path_effects=STROKE,
    )

# ── Panel labels ─────────────────────────────────────────────────────────────

ax1.minorticks_on()
ax2.minorticks_on()
ax1.text(-0.20, 0.98, "(a)", transform=ax1.transAxes,
         fontsize=12, va="top", ha="left")
ax2.text(-0.20, 0.98, "(b)", transform=ax2.transAxes,
         fontsize=12, va="top", ha="left")

# ── Manuscript numbers ───────────────────────────────────────────────────────


def group_dev(comps):
    """RMS deviation over a component subset; returns (curve, argmax index)."""
    d = np.sqrt(sum(per[c] ** 2 for c in comps) / len(comps))
    return d, int(np.argmax(d))


dev_b, ib = group_dev(("n_ixx", "n_izz"))
dev_d, idx_d = group_dev(("n_xx", "n_zz"))
_i37 = int(np.argmin(np.abs(grid - E_HIQ)))

print("--- manuscript numbers ---")
print(
    f"beta : max eps = {dev_b[ib]:.2f}% at {grid[ib]:.2f} eV | "
    f"mean = {dev_b.mean():.2f}% | at 283.7 eV = {dev_b[_i37]:.2f}%"
)
print(
    f"delta: max eps = {dev_d[idx_d]:.2f}% at {grid[idx_d]:.2f} eV | "
    f"mean = {dev_d.mean():.2f}% | at 283.7 eV = {dev_d[_i37]:.2f}%"
)
print(
    f"total: max eps = {eps[_imax]:.2f}% at {grid[_imax]:.2f} eV | "
    f"mean over {lb}-{ub} eV = {eps.mean():.2f}%"
)
for _c, _v in per.items():
    print(
        f"  {_c}: {_v[_i37]:+.2f}% at 283.7 eV, "
        f"max |dev| = {np.max(np.abs(_v)):.2f}% "
        f"at {grid[int(np.argmax(np.abs(_v)))]:.2f} eV"
    )

print("--- caption sentence (LaTeX) ---")
print(
    f"The DFT-derived tensor reproduces the experimental optical constants "
    f"to within {eps.mean():.1f}\\% on average over {lb}--{ub}\\,eV, with "
    f"maximum deviations of {dev_b[ib]:.1f}\\% in $\\beta$ at "
    f"{grid[ib]:.1f}\\,eV and {dev_d[idx_d]:.1f}\\% in $\\delta$ at "
    f"{grid[idx_d]:.1f}\\,eV."
)

fig.align_labels()
fig.tight_layout()
fig.savefig(
    "FIG1.png", dpi=600, bbox_inches="tight",
    transparent=True,
)
fig.savefig(
    "FIG1.pdf", dpi=600, bbox_inches="tight",
    transparent=True,
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
import matplotlib.patheffects as pe
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

plt.rcParams["hatch.linewidth"] = 0.5
STROKE = [pe.Stroke(linewidth=1.5, foreground="white"), pe.Normal()]

lb, ub = 281, 290
E_HIQ = 283.7
PURPLE = "#8250DF"

# ── Dial ranges ──────────────────────────────────────────────────────────────
rho = np.array([0.75, 0.9, 1.0, 1.25, 1.5, 2.0])        # density factor
gam = np.deg2rad(np.array([0, 10, 20, 30, 40, 50]))     # orientation factor

def density_scale(component, density):
    return component * density

def rotate_tensor(xx, zz, gamma):
    xx_r = (xx * (1 + np.cos(gamma) ** 2) + zz * np.sin(gamma) ** 2) / 2
    zz_r = xx * np.sin(gamma) ** 2 + zz * np.cos(gamma) ** 2
    return xx_r, zz_r

def density_band(comp):
    """Min/max envelope of a component swept over the density dial."""
    stack = np.outer(rho, comp)
    return stack.min(axis=0), stack.max(axis=0)

def orient_band(xx, zz):
    """Min/max envelopes of both components swept over the tilt dial."""
    xr = np.array([rotate_tensor(xx, zz, g)[0] for g in gam])
    zr = np.array([rotate_tensor(xx, zz, g)[1] for g in gam])
    return (xr.min(0), xr.max(0)), (zr.min(0), zr.max(0))

def interp_at(e, y, val):
    return np.interp(val, e, y)

# ── Data ─────────────────────────────────────────────────────────────────────
d = dft[(dft["energy"] >= lb) & (dft["energy"] <= ub)]
e = d["energy"].to_numpy()
b_xx, b_zz = d["n_ixx"].to_numpy(), d["n_izz"].to_numpy()   # beta traces
d_xx, d_zz = d["n_xx"].to_numpy(),  d["n_zz"].to_numpy()    # delta traces

# ── Panel drawer ─────────────────────────────────────────────────────────────
def draw_dials(ax, xx, zz, clip_to_nominal=True):
    l0, = ax.plot(e, xx, "-", color="C0", lw=1.0, zorder=3)
    l1, = ax.plot(e, zz, "-", color="C1", lw=1.0, zorder=3)
    yl = ax.get_ylim()
    for comp, c in ((xx, "C0"), (zz, "C1")):        # density, solid tint
        lo, hi = density_band(comp)
        ax.fill_between(e, lo, hi, facecolor=c, alpha=0.16, lw=0, zorder=-12)
    (xlo, xhi), (zlo, zhi) = orient_band(xx, zz)    # orientation, hatch
    ax.fill_between(e, xlo, xhi, facecolor="none", edgecolor="C0",
                    hatch="////", lw=0, zorder=-11)
    ax.fill_between(e, zlo, zhi, facecolor="none", edgecolor="C1",
                    hatch="////", lw=0, zorder=-11)
    if clip_to_nominal:
        ax.set_ylim(yl)
    return l0, l1

# ── Figure ───────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(3.35, 4.5), sharex=True, dpi=300,
    gridspec_kw={"hspace": 0.1},
)

draw_dials(ax1, b_xx, b_zz, clip_to_nominal=True)   # beta
draw_dials(ax2, d_xx, d_zz, clip_to_nominal=False)  # delta
ax2.axhline(0, color=plt.rcParams["axes.edgecolor"],
            lw=plt.rcParams["axes.linewidth"])

# ── Probe energies (one channel, both panels) ────────────────────────────────
for ax in (ax1, ax2):
    ax.axvline(E_HIQ, color=PURPLE, lw=0.8, ls=(0, (3, 2)), alpha=0.9, zorder=-5)

comb_tr = mtransforms.blended_transform_factory(ax1.transData, ax1.transAxes)
for en in pxr:
    if E_HIQ < en <= ub:
        ax1.plot([en, en], [1.0, 0.955], transform=comb_tr, color="0.3",
                 lw=0.8, solid_capstyle="butt", clip_on=False, zorder=5)
        ax1.vlines(en, interp_at(e, b_xx, en), interp_at(e, b_zz, en),
                   color="0.65", lw=0.8, zorder=-8)
        ax2.vlines(en, interp_at(e, d_xx, en), interp_at(e, d_zz, en),
                   color="0.65", lw=0.8, zorder=-8)

ax1.annotate(f"{E_HIQ} eV",
             xy=(E_HIQ, 0.97), xytext=(2.5, 0), textcoords="offset points",
             xycoords=mtransforms.blended_transform_factory(ax1.transData, ax1.transAxes),
             rotation=90, ha="left", va="top", fontsize=6.5,
             color=PURPLE, path_effects=STROKE, zorder=6)

ax1.annotate("250 eV", xy=(0.012, 0.12), xytext=(0.14, 0.12),
             xycoords="axes fraction", textcoords="axes fraction",
             ha="left", va="center", fontsize=7, color="0.25",
             arrowprops=dict(arrowstyle="-|>", color="0.25", lw=0.8))

# ── Dial-range key ───────────────────────────────────────────────────────────
ax2.text(0.97, 0.05,
         r"$\rho \in [0.75,\,2.0]$" "\n" r"$\gamma \in [0^\circ,\,50^\circ]$",
         transform=ax2.transAxes, fontsize=7, ha="right", va="bottom",
         color="0.3", linespacing=1.4, path_effects=STROKE)

# ── Cosmetics ────────────────────────────────────────────────────────────────
ax1.set_ylabel(r"$\beta$ (a.u.)")
ax2.set_ylabel(r"$\delta$ (a.u.)")
ax2.set_xlabel("Energy (eV)")
ax1.set_xlim(lb, ub)
for ax in (ax1, ax2):
    ax.ticklabel_format(style="sci", axis="y", scilimits=(0, 0))
    ax.yaxis.offsetText.set_fontsize(10)
    ax.tick_params(axis="both", which="major", labelsize=11)
    ax.minorticks_on()

handles = [
    Line2D([0], [0], color="C0", lw=1, label=r"$n_{xx}$"),
    Line2D([0], [0], color="C1", lw=1, label=r"$n_{zz}$"),
    Patch(facecolor="0.5", alpha=0.16, label="density sweep"),
    Patch(facecolor="none", edgecolor="0.4", hatch="////", label="tilt sweep"),
    Line2D([0], [0], color=PURPLE, lw=0.8, ls=(0, (3, 2)), label="283.7 eV long-q"),
    Line2D([0], [0], color="0.65", lw=0.8, label="tensor contrast"),
]
ax1.legend(handles=handles, loc="upper right", frameon=False,
           fontsize=6.5, labelspacing=0.25, handlelength=1.2)

ax1.text(-0.20, 0.98, "(a)", transform=ax1.transAxes, fontsize=12, va="top")
ax2.text(-0.20, 0.98, "(b)", transform=ax2.transAxes, fontsize=12, va="top")

fig.align_labels()
fig.tight_layout()
fig.savefig("fig1_dials_probe_energies.png", dpi=600,
            bbox_inches="tight", transparent=True)
fig.savefig("fig1_dials_probe_energies.pdf", dpi=600,
            bbox_inches="tight", transparent=True)